# Análisis Distribuido de Auditoría de Activos de Energía Renovable con Apache Spark

**Especialización en Inteligencia Artificial**

## Objetivo
Aplicar los principios de procesamiento distribuido para limpiar, transformar y analizar un dataset de **1,200,000 registros** de telemetría de activos de generación renovable con **PySpark**, identificando patrones de degradación, fallas y rendimiento operativo.

## Dataset
`auditoria_activos_renovables.csv` — Lecturas de sensores y diagnósticos de activos de generación renovable en Colombia.

| Columna | Tipo | Descripción |
|---|---|---|
| `uuid_evento` | string (UUID) | Identificador único de cada lectura |
| `codigo_activo` | int | Código del equipo físico (500000–999999) |
| `tecnologia_generacion` | string | `Fotovoltaica_Tracking`, `Fotovoltaica_Fija`, `Aerogenerador_Onshore` |
| `nodo_inyeccion` | string | Subestación de conexión a la red eléctrica |
| `energia_inyectada_mwh` | double | Energía entregada al sistema (MWh) |
| `performance_ratio` | double | Ratio de rendimiento real vs teórico (0.50–0.98) |
| `perdida_tecnica_pct` | double | Pérdida técnica en porcentaje (1.2%–14.5%) |
| `vibracion_cojinete_mms` | double | Vibración del cojinete (mm/s) — crítica en aerogeneradores |
| `temperatura_devanado_c` | double | Temperatura del devanado del generador (°C) |
| `diagnostico_salud` | string | `Optimo`, `Degradacion_Leve`, `Falla_Termica`, `Desalineacion_Mecanica` |
| `horas_acumuladas_servicio` | int | Horas totales de operación del activo |
| `empresa_certificadora` | string | Contratista que realizó la certificación |
| `timestamp_medicion` | string → timestamp | Marca de tiempo de la medición |

>  El dataset fue generado sintéticamente con **1.2 millones de registros**. Las distribuciones estadísticas son realistas: vibración con distribución Gamma para aerogeneradores, temperatura con distribución normal, performance ratio con clip [0.50, 0.98]. Pueden existir valores atípicos en temperatura y vibración que representan condiciones de falla.

---
## 1. Configuración del Entorno Spark

Levantamos una `SparkSession` ajustada al volumen del dataset.

**Conceptos clave:**
- `spark.driver.memory` → memoria del proceso *driver* (coordinador). En Colab no hay cluster, el driver hace todo.
- `spark.sql.shuffle.partitions` → número de particiones tras un `groupBy`/`join`. El default es **200**, excesivo para datos locales. Regla: `núcleos × 2`.
- `spark.sql.adaptive.enabled` (**AQE**) → reorganiza el plan de ejecución en runtime: combina particiones pequeñas, mitiga *skew* y elige el mejor tipo de join.
- **Esquema explícito** evita una pasada extra sobre el archivo (`inferSchema` lee dos veces).
- **`cache()`** mantiene el DataFrame en memoria entre acciones — clave cuando se hacen 10+ consultas sobre el mismo DF.

In [ ]:
# Si trabajas en Google Colab, descomenta:
# !pip install pyspark==3.5.0 -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)

spark = (
    SparkSession.builder
    .appName("AuditoriaActivosRenovables")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
spark

---
## 2. Carga del Dataset

Se define un **esquema explícito** para evitar la doble lectura de `inferSchema`. El `timestamp_medicion` se carga como `StringType` y se parsea inmediatamente a `TimestampType` con `to_timestamp`.

> Si el archivo está en Google Drive:
> ```python
> from google.colab import drive; drive.mount('/content/drive')
> RUTA = '/content/drive/MyDrive/auditoria_activos_renovables.csv'
> ```

In [ ]:
schema = StructType([
    StructField("uuid_evento",              StringType(),  True),
    StructField("codigo_activo",            IntegerType(), True),
    StructField("tecnologia_generacion",    StringType(),  True),
    StructField("nodo_inyeccion",           StringType(),  True),
    StructField("energia_inyectada_mwh",    DoubleType(),  True),
    StructField("performance_ratio",        DoubleType(),  True),
    StructField("perdida_tecnica_pct",      DoubleType(),  True),
    StructField("vibracion_cojinete_mms",   DoubleType(),  True),
    StructField("temperatura_devanado_c",   DoubleType(),  True),
    StructField("diagnostico_salud",        StringType(),  True),
    StructField("horas_acumuladas_servicio",IntegerType(), True),
    StructField("empresa_certificadora",    StringType(),  True),
    StructField("timestamp_medicion",       StringType(),  True),
])

RUTA = "auditoria_activos_renovables.csv"   # <-- ajusta esta ruta

df = (
    spark.read
    .option("header", True)
    .option("nullValue", "")
    .schema(schema)
    .csv(RUTA)
    .withColumn("timestamp_medicion", F.to_timestamp("timestamp_medicion", "yyyy-MM-dd HH:mm:ss"))
)

df.cache()
total_registros = df.count()
print(f"Total de registros: {total_registros:,}")
df.printSchema()
df.show(5, truncate=False)

### Exploración rápida
Antes de limpiar, una mirada general: tipos, nulos por columna y muestra de distribuciones.

In [ ]:
# Conteo de nulos por columna
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
]).show()

---
## Pregunta 1 — Integridad de Eventos (Duplicados)

> *¿Cuántos registros duplicados existen en el dataset? Elimínalos y crea un DataFrame limpio base.*

**Conceptos:**
- `df.distinct()` — elimina filas idénticas en **todas** las columnas.
- `df.dropDuplicates([cols])` — permite indicar un subconjunto (ej. `["uuid_evento"]`). Dado que cada UUID debería ser único, también puedes deduplicar por esa columna.
- Ambos disparan un **shuffle** (operación costosa en distribuido).
- El conteo de duplicados se obtiene como: `total − total_único`.

**Tu tarea:**
1. Calcular cuántos registros son duplicados.
2. Crear un nuevo DataFrame `df_clean` ya sin duplicados y persistirlo con `.cache()`.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista:
# total = df.count()
# df_clean = df.dropDuplicates()
# df_clean.cache()
# unicos = df_clean.count()
# print(f"Duplicados eliminados: {total - unicos:,}")


---
##  Pregunta 2 — Tratamiento de Nulos en Columnas Críticas

> *Identifica cuántos nulos hay en cada columna. Para las columnas numéricas críticas (`energia_inyectada_mwh`, `performance_ratio`, `temperatura_devanado_c`), decide una estrategia: ¿eliminar filas o imputar con la mediana por tecnología?*

**Conceptos:**
- `df.dropna(subset=[...])` — elimina filas con nulos en las columnas indicadas.
- Imputación por mediana: calcular la mediana por grupo con `F.percentile_approx(col, 0.5)` y luego usar un `join` + `coalesce` para rellenar.
- `F.coalesce(col_original, col_mediana)` — devuelve el primer valor no-nulo.
- La mediana es más robusta que la media ante outliers (ej. temperaturas extremas).

**Tu tarea:**
1. Mostrar el conteo de nulos por columna.
2. Aplicar `dropna` o imputación sobre `df_clean` y reportar cuántas filas se afectaron.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista (estrategia simple — eliminar):
# antes = df_clean.count()
# df_clean = df_clean.dropna(subset=["energia_inyectada_mwh", "performance_ratio", "temperatura_devanado_c"])
# despues = df_clean.count()
# print(f"Filas eliminadas por nulos críticos: {antes - despues:,}")


---
##  Pregunta 3 — Detección de Outliers en Temperatura del Devanado

> *La temperatura del devanado sigue una distribución normal con μ=68°C y σ=14°C. Identifica las lecturas fuera del rango [μ − 3σ, μ + 3σ] = [26°C, 110°C]. ¿Cuántas lecturas son outliers? ¿Qué diagnóstico de salud predomina en ellas?*

**Conceptos:**
- Regla de 3-sigma: en una distribución normal, el 99.7% de los datos cae dentro de μ ± 3σ.
- `F.col("temperatura_devanado_c").between(26, 110)` filtra el rango.
- Para un análisis más dinámico, puedes calcular μ y σ con `F.avg()` y `F.stddev()` en vez de hardcodear.
- Cruzar outliers con `diagnostico_salud` ayuda a validar si las fallas térmicas se reflejan en los diagnósticos.

**Tu tarea:**
1. Filtrar los registros fuera del rango [26, 110] °C.
2. Contar cuántos son y mostrar su distribución por `diagnostico_salud`.
3. Decidir si eliminarlos o marcarlos.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista:
# outliers_temp = df_clean.filter(
#     ~F.col("temperatura_devanado_c").between(26, 110)
# )
# print(f"Outliers de temperatura: {outliers_temp.count():,}")
# outliers_temp.groupBy("diagnostico_salud").count().orderBy(F.desc("count")).show()


---
## Pregunta 4 — Energía Inyectada por Tecnología

> *Calcula la energía total, promedio y máxima inyectada por cada `tecnologia_generacion`. ¿Cuál contribuye más al sistema?*

**Conceptos:**
- `groupBy("tecnologia_generacion").agg(...)` — múltiples agregaciones en una sola pasada.
- `F.sum()`, `F.avg()`, `F.max()`, `F.stddev()` — funciones de agregación.
- `F.round(expr, n)` para limitar decimales en la salida.
- Este es el patrón básico **groupBy → agg → orderBy** que se repite en Big Data.

**Tu tarea:**
1. Agrupar por tecnología de generación.
2. Calcular: `count`, `sum`, `avg`, `max` de `energia_inyectada_mwh`.
3. Ordenar por energía total descendente.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista:
# resumen_energia = (df_clean
#     .groupBy("tecnologia_generacion")
#     .agg(
#         F.count("*").alias("total_lecturas"),
#         F.round(F.sum("energia_inyectada_mwh"), 2).alias("energia_total_mwh"),
#         F.round(F.avg("energia_inyectada_mwh"), 4).alias("energia_promedio_mwh"),
#         F.round(F.max("energia_inyectada_mwh"), 4).alias("energia_maxima_mwh"),
#     )
#     .orderBy(F.desc("energia_total_mwh")))
# resumen_energia.show(truncate=False)


---
##  Pregunta 5 — Performance Ratio por Subestación

> *¿Cuál subestación (`nodo_inyeccion`) presenta el mejor y el peor rendimiento promedio (`performance_ratio`)? ¿Existe correlación con las pérdidas técnicas?*

**Conceptos:**
- `performance_ratio` mide qué porcentaje de la energía teórica se entrega realmente. Valores cercanos a 1 son óptimos.
- `perdida_tecnica_pct` debería tener correlación inversa con el PR: a mayor pérdida, menor rendimiento.
- Para validar la correlación, puedes usar `F.corr(col1, col2)` que calcula el coeficiente de Pearson.

**Tu tarea:**
1. Agrupar por `nodo_inyeccion` y calcular `avg(performance_ratio)` y `avg(perdida_tecnica_pct)`.
2. Calcular la correlación global entre ambas variables.
3. Identificar la mejor y peor subestación.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista:
# pr_por_nodo = (df_clean
#     .groupBy("nodo_inyeccion")
#     .agg(
#         F.round(F.avg("performance_ratio"), 4).alias("pr_promedio"),
#         F.round(F.avg("perdida_tecnica_pct"), 2).alias("perdida_prom_pct"),
#         F.count("*").alias("lecturas")
#     )
#     .orderBy(F.desc("pr_promedio")))
# pr_por_nodo.show()
#
# # Correlación global
# corr = df_clean.stat.corr("performance_ratio", "perdida_tecnica_pct")
# print(f"Correlación PR vs Pérdida Técnica: {corr:.4f}")


---
## Pregunta 6 — Diagnóstico de Salud por Tecnología (Tabla Cruzada)

> *Construye una tabla cruzada de `tecnologia_generacion` vs `diagnostico_salud`. ¿Alguna tecnología concentra más fallas térmicas o desalineaciones mecánicas?*

**Conceptos:**
- `groupBy(col_fila).pivot(col_columna).count()` — genera la tabla dinámica directamente en Spark.
- `pivot` convierte cada valor único de `diagnostico_salud` en una columna.
- La distribución original fue: 83% Óptimo, 11% Degradación Leve, 4% Falla Térmica, 2% Desalineación Mecánica.
- Un desvío significativo respecto a esas proporciones en alguna tecnología indicaría un patrón de falla específico.

**Tu tarea:**
1. Generar la tabla cruzada tecnología vs diagnóstico.
2. Interpretar si alguna tecnología muestra concentración anómala de fallas.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista:
# tabla_cruzada = (df_clean
#     .groupBy("tecnologia_generacion")
#     .pivot("diagnostico_salud")
#     .count()
#     .orderBy("tecnologia_generacion"))
# tabla_cruzada.show(truncate=False)


---
## Pregunta 7 — Análisis de Vibración en Aerogeneradores

> *Filtra únicamente los registros de `Aerogenerador_Onshore`. ¿Cuál es la vibración promedio, máxima y el percentil 95? ¿Cuántos registros superan el umbral crítico de 5.0 mm/s?*

**Conceptos:**
- La vibración de cojinetes en aerogeneradores sigue una **distribución Gamma** (α=2.5, β=1.2) en este dataset. Es asimétrica con cola larga a la derecha.
- `F.percentile_approx(col, 0.95)` — calcula el percentil 95 de forma aproximada (eficiente en Big Data).
- En la industria eólica, vibraciones >5 mm/s suelen disparar alertas de mantenimiento.
- Patrón: **filter → agg** para focalizar el análisis en un subconjunto.

**Tu tarea:**
1. Filtrar por `Aerogenerador_Onshore`.
2. Calcular `avg`, `max`, `percentile_approx(0.95)` de `vibracion_cojinete_mms`.
3. Contar cuántos superan 5.0 mm/s y qué `diagnostico_salud` predomina en ellos.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista:
# aero = df_clean.filter(F.col("tecnologia_generacion") == "Aerogenerador_Onshore")
#
# aero.agg(
#     F.round(F.avg("vibracion_cojinete_mms"), 3).alias("vib_promedio"),
#     F.round(F.max("vibracion_cojinete_mms"), 3).alias("vib_maxima"),
#     F.round(F.percentile_approx("vibracion_cojinete_mms", 0.95), 3).alias("vib_p95"),
# ).show()
#
# criticos = aero.filter(F.col("vibracion_cojinete_mms") > 5.0)
# print(f"Registros con vibración crítica (>5 mm/s): {criticos.count():,}")
# criticos.groupBy("diagnostico_salud").count().orderBy(F.desc("count")).show()


---
##  Pregunta 8 — Clasificación de Vida Útil del Activo

> *Crea una nueva columna `etapa_vida` que clasifique cada activo según sus `horas_acumuladas_servicio`:*
>
> | Rango de horas | Etapa |
> |---|---|
> | < 5,000 | Rodaje |
> | 5,000 – 20,000 | Vida_Util |
> | 20,000 – 35,000 | Desgaste |
> | > 35,000 | Fin_Vida |
>
> *¿Cómo se distribuyen los diagnósticos de salud en cada etapa?*

**Conceptos:**
- `F.when(...).when(...).otherwise(...)` — equivalente a un `CASE WHEN` de SQL.
- Las etapas reflejan la **curva de bañera** de confiabilidad: alta tasa de fallo inicial (rodaje), baja en vida útil, creciente en desgaste/fin de vida.
- Cruzar `etapa_vida` con `diagnostico_salud` permite validar si la tasa de falla sigue ese patrón.

**Tu tarea:**
1. Crear la columna `etapa_vida` con `withColumn` + `when`.
2. Mostrar conteo de lecturas por etapa.
3. Cruzar `etapa_vida` vs `diagnostico_salud` para analizar la tendencia.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista:
# df_clean = df_clean.withColumn(
#     "etapa_vida",
#     F.when(F.col("horas_acumuladas_servicio") < 5000, "Rodaje")
#      .when(F.col("horas_acumuladas_servicio") < 20000, "Vida_Util")
#      .when(F.col("horas_acumuladas_servicio") < 35000, "Desgaste")
#      .otherwise("Fin_Vida")
# )
#
# df_clean.groupBy("etapa_vida").count().orderBy("etapa_vida").show()
#
# (df_clean
#     .groupBy("etapa_vida")
#     .pivot("diagnostico_salud")
#     .count()
#     .orderBy("etapa_vida")
#     .show(truncate=False))


---
##  Pregunta 9 — Top Empresas Certificadoras

> *Identifica las **10 empresas certificadoras** que más lecturas registran. De ellas, ¿cuál tiene la mayor proporción de diagnósticos `Falla_Termica` o `Desalineacion_Mecanica`?*

**Conceptos:**
- Top-N → `orderBy(F.desc("count")).limit(10)`.
- Para calcular la tasa de falla por empresa en una sola pasada:
  ```python
  .agg(
      F.count("*").alias("total"),
      F.sum(
          F.col("diagnostico_salud").isin("Falla_Termica", "Desalineacion_Mecanica").cast("int")
      ).alias("fallas")
  )
  ```
- La tasa = `fallas / total * 100`.
- Una empresa con tasa de falla significativamente mayor podría indicar baja calidad de certificación.

**Tu tarea:**
1. Agrupar por `empresa_certificadora`, calcular total y fallas.
2. Calcular el porcentaje de fallas.
3. Limitar al Top 10 más activas y ordenar por tasa de falla.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista:
# top_empresas = (df_clean
#     .groupBy("empresa_certificadora")
#     .agg(
#         F.count("*").alias("total_lecturas"),
#         F.sum(
#             F.col("diagnostico_salud").isin("Falla_Termica", "Desalineacion_Mecanica").cast("int")
#         ).alias("fallas")
#     )
#     .withColumn("pct_falla", F.round(F.col("fallas") / F.col("total_lecturas") * 100, 2))
#     .orderBy(F.desc("total_lecturas"))
#     .limit(10))
# top_empresas.orderBy(F.desc("pct_falla")).show(truncate=False)


---
##  Pregunta 10 — Análisis Temporal de la Generación

> *Agrupa los datos por **día** extrayendo la fecha del `timestamp_medicion`. ¿Cuál fue el día con mayor energía total inyectada y cuál con más eventos de falla?*

**Conceptos:**
- `F.to_date(col)` o `F.date_format(col, "yyyy-MM-dd")` extrae la parte de fecha.
- `F.dayofweek(col)` devuelve 1 (Domingo) a 7 (Sábado) — útil para detectar patrones semanales.
- El dataset cubre ~60 días con lecturas cada 2 segundos, lo que da ~20,000 lecturas/día.

**Tu tarea:**
1. Extraer la fecha en una nueva columna.
2. Agrupar por fecha y calcular: `sum(energia_inyectada_mwh)` y conteo de fallas (`Falla_Termica` + `Desalineacion_Mecanica`).
3. Mostrar los top 5 días con más energía y los top 5 con más fallas.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista:
# df_temporal = df_clean.withColumn("fecha", F.to_date("timestamp_medicion"))
#
# por_dia = (df_temporal
#     .groupBy("fecha")
#     .agg(
#         F.round(F.sum("energia_inyectada_mwh"), 2).alias("energia_total_mwh"),
#         F.sum(
#             F.col("diagnostico_salud").isin("Falla_Termica", "Desalineacion_Mecanica").cast("int")
#         ).alias("total_fallas"),
#         F.count("*").alias("lecturas")
#     ))
#
# print("=== Top 5 días con mayor energía ===")
# por_dia.orderBy(F.desc("energia_total_mwh")).show(5)
#
# print("=== Top 5 días con más fallas ===")
# por_dia.orderBy(F.desc("total_fallas")).show(5)


---
## 3. Exportación del Dataset Limpio

**Parquet vs CSV:**
- **Parquet** → columnar, comprimido, con esquema embebido. Lectura selectiva por columnas. **Recomendado** para Big Data.
- **CSV** → universal pero pesado y sin tipos. Solo si necesitas abrirlo en Excel.

`coalesce(1)` fuerza la salida a un único archivo (a costa de perder paralelismo). Para producción se prefiere dejar Spark elegir el número de particiones.

In [ ]:
# Resumen agregado para visualización rápida
resumen = (df_clean
    .groupBy("nodo_inyeccion", "tecnologia_generacion", "diagnostico_salud")
    .agg(
        F.count("*").alias("total_eventos"),
        F.round(F.sum("energia_inyectada_mwh"), 2).alias("energia_total_mwh"),
        F.round(F.avg("performance_ratio"), 4).alias("pr_promedio"),
        F.round(F.avg("temperatura_devanado_c"), 2).alias("temp_promedio")
    ))

# Dataset limpio completo en Parquet (recomendado)
df_clean.write.mode("overwrite").parquet("auditoria_clean.parquet")

# Resumen en CSV (para abrir en Excel/Power BI)
(resumen.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv("auditoria_resumen.csv"))

print(" Exportación completa")

---
## 4. Cierre de la sesión Spark

Liberar recursos siempre — sobre todo si vas a re-ejecutar el notebook.

In [ ]:
df.unpersist()
df_clean.unpersist()
spark.stop()

---
## Checklist de Entrega

- [ ] Notebook ejecutado de principio a fin sin errores.
- [ ] Las **10 preguntas** respondidas con su captura de salida.
- [ ] Archivo `auditoria_clean.parquet` generado.
- [ ] Archivo `auditoria_resumen.csv` generado.